In [ ]:
# ========== 1. IMPORTS ========== #
import os
import csv
import time
import random
import shutil
import glob
import gc
import tempfile
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm_lib
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, Activation, Input,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger, Callback
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import load_img, img_to_array

from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    top_k_accuracy_score, roc_curve, auc,
    cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ========== 2. GPU CONFIG ========== #
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", len(gpus))
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        print(e)

# ========== 3. MIXED PRECISION ========== #
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Compute dtype :", tf.keras.mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", tf.keras.mixed_precision.global_policy().variable_dtype)

In [ ]:
# ========== CUSTOM LOSS: ADAPTIVE CB FOCAL LOSS v2 (2-Phase) ========== #

class AdaptiveClassBalancedFocalLossV2(tf.keras.losses.Loss):
    """Adaptive Class-Balanced Focal Loss — Version 2.
    
    Key differences from v1:
    - beta=0.999 (softer weight gap for moderate imbalance)
    - gamma=1.5 (less aggressive focal modulation)
    - adaptive_factor controlled externally by callback (2-phase)
    - Added label smoothing (0.1) for additional regularization
    """
    def __init__(self, samples_per_class, num_classes, gamma=1.5, beta=0.999,
                 label_smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.gamma           = gamma
        self.beta            = beta
        self.num_classes     = num_classes
        self.label_smoothing = label_smoothing
        self._samples_per_class = list(samples_per_class)
        
        # Static CB weights
        n       = np.array(samples_per_class, dtype=np.float32)
        eff_num = 1.0 - np.power(beta, n)
        weights = (1.0 - beta) / eff_num
        weights = weights / weights.sum() * len(samples_per_class)
        self.static_weights = tf.constant(weights, dtype=tf.float32)
        
        # Adaptive factor — starts at 1.0, updated by callback only in Phase 2
        self.adaptive_factor = tf.Variable(
            tf.ones([num_classes], dtype=tf.float32),
            trainable=False, name='adaptive_cb_factor_v2'
        )
        
        # Phase flag — callback sets this to True when Phase 2 starts
        self.adaptation_active = tf.Variable(
            False, trainable=False, name='adaptation_active'
        )

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        
        # Label smoothing
        if self.label_smoothing > 0:
            y_true = y_true * (1.0 - self.label_smoothing) + \
                     self.label_smoothing / tf.cast(tf.shape(y_true)[-1], tf.float32)
        
        # Weight selection: static only in Phase 1, static*adaptive in Phase 2
        combined_weights = tf.cond(
            self.adaptation_active,
            lambda: self.static_weights * self.adaptive_factor,
            lambda: self.static_weights,
        )
        # Normalize
        combined_weights = combined_weights / tf.reduce_mean(combined_weights)
        
        # Per-sample weight
        sample_w = tf.reduce_sum(y_true * combined_weights, axis=-1)
        
        # Focal modulation (gamma=1.5, softer than standard 2.0)
        pt    = tf.reduce_sum(y_true * y_pred, axis=-1)
        focal = tf.pow(1.0 - pt, self.gamma)
        
        # Cross-entropy
        ce = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)
        
        return tf.reduce_mean(sample_w * focal * ce)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'samples_per_class': self._samples_per_class,
            'num_classes': self.num_classes,
            'gamma': self.gamma,
            'beta': self.beta,
            'label_smoothing': self.label_smoothing,
        })
        return cfg


class TwoPhaseAdaptiveCallback(Callback):
    """2-Phase Adaptive Weight Callback.
    
    Phase 1 (epochs 1 to phase1_epochs): Static CB weights only.
        - Model learns basic features with stable gradients.
        - No adaptation — just monitors per-class recall.
    
    Phase 2 (epochs phase1_epochs+1 to end): Adaptive weights activated.
        - Per-class recall feedback loop with conservative EMA (tau=0.15).
        - Only adjusts weights when recall gap > threshold (stability guard).
    """
    def __init__(self, loss_fn, val_ds, num_classes, class_names,
                 phase1_epochs=10, tau=0.15, min_recall_gap=0.05, **kwargs):
        super().__init__(**kwargs)
        self.loss_fn        = loss_fn
        self.val_ds         = val_ds
        self.num_classes    = num_classes
        self.class_names    = class_names
        self.phase1_epochs  = phase1_epochs
        self.tau            = tau
        self.min_recall_gap = min_recall_gap  # Only adapt if max-min recall > this
        self.history        = []
    
    def on_epoch_end(self, epoch, logs=None):
        # Predict on validation set
        y_pred_probs = self.model.predict(self.val_ds, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in self.val_ds])
        
        # Per-class recall
        per_class_recall = np.zeros(self.num_classes)
        for c in range(self.num_classes):
            mask = y_true == c
            if mask.sum() > 0:
                per_class_recall[c] = (y_pred[mask] == c).mean()
            else:
                per_class_recall[c] = 1.0
        
        current_factor = self.loss_fn.adaptive_factor.numpy()
        phase = "Phase 1 (static)" if (epoch + 1) <= self.phase1_epochs else "Phase 2 (adaptive)"
        
        # Phase 2: activate adaptation
        if (epoch + 1) == self.phase1_epochs + 1:
            self.loss_fn.adaptation_active.assign(True)
            print(f"\n  >>> PHASE 2 ACTIVATED at epoch {epoch+1} <<<")
        
        if (epoch + 1) > self.phase1_epochs:
            # Stability guard: only adapt if recall gap is significant
            recall_gap = per_class_recall.max() - per_class_recall.min()
            
            if recall_gap > self.min_recall_gap:
                # Adaptation target: inverse recall (lower recall → higher weight)
                epsilon = 0.15
                adaptation_target = (1.0 - per_class_recall) + epsilon
                
                # Conservative EMA update
                new_factor = (1.0 - self.tau) * current_factor + self.tau * adaptation_target
                
                # Normalize (mean = 1)
                new_factor = new_factor / new_factor.mean()
                
                # Clamp to prevent extreme values [0.5, 2.0]
                new_factor = np.clip(new_factor, 0.5, 2.0)
                new_factor = new_factor / new_factor.mean()  # re-normalize after clip
                
                self.loss_fn.adaptive_factor.assign(new_factor.astype(np.float32))
                current_factor = new_factor
                adapted = True
            else:
                adapted = False
                
            status = f"adapted (gap={recall_gap:.3f})" if adapted else f"skipped (gap={recall_gap:.3f}<{self.min_recall_gap})"
        else:
            status = "monitoring only"
        
        self.history.append({
            'epoch': epoch + 1,
            'phase': phase,
            'per_class_recall': per_class_recall.copy(),
            'adaptive_factor': current_factor.copy(),
        })
        
        print(f"  [AdaptiveCB-v2] Epoch {epoch+1} | {phase} | {status}")
        print(f"    Recall : {dict(zip(self.class_names, [f'{r:.3f}' for r in per_class_recall]))}")
        print(f"    Factors: {dict(zip(self.class_names, [f'{f:.3f}' for f in current_factor]))}")


print("AdaptiveClassBalancedFocalLossV2 defined")
print("  - 2-Phase: static warmup → adaptive fine-tuning")
print("  - Conservative tau=0.15 with stability guard")
print("  - Factor clamping [0.5, 2.0] prevents extreme weights")
print("  - Label smoothing 0.1 for regularization")

In [ ]:
# ========== 4. EXPERIMENT CONFIGURATION ========== #

# ── STRATEGY ──────────────────────────────────────────────────────────────
STRATEGY_KEY    = "finetune_top5_fsda_adaptive_cb_v2"
STRATEGY_LABEL  = "EfficientNetB4 + FSDA + Adaptive CB Loss v2 (2-Phase)"
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]
USE_AUG         = True
LR              = 1e-4
# ──────────────────────────────────────────────────────────────────────────

# ── FSDA HYPERPARAMETERS ──────────────────────────────────────────────────
FSDA_REDUCTION    = 16
FSDA_SPATIAL_KS   = 7
# ──────────────────────────────────────────────────────────────────────────

# ── ADAPTIVE LOSS v2 HYPERPARAMETERS ──────────────────────────────────────
ADAPTIVE_TAU      = 0.15    # Conservative EMA (was 0.3 in v1)
FOCAL_GAMMA       = 1.5     # Softer focal (was 2.0 in v1)
CB_BETA           = 0.999   # Better for moderate imbalance (was 0.9999)
LABEL_SMOOTHING   = 0.1     # Additional regularization
PHASE1_EPOCHS     = 10      # Static CB warmup phase
MIN_RECALL_GAP    = 0.05    # Stability guard threshold
# ──────────────────────────────────────────────────────────────────────────

# --- Paths ---
DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# --- Model ---
INPUT_SHAPE = (380, 380, 3)
BATCH_SIZE  = 32
EPOCHS      = 40      # Extended (was 30 in v1)

# --- Shared hyperparameters ---
DROPOUT_RATE    = 0.4   # Reduced (was 0.5) — FSDA already regularizes
PATIENCE        = 15    # Extended (was 12)

# --- Multi-run settings ---
N_RUNS       = 3
RANDOM_SEEDS = [42, 123, 456]

# --- Performance knobs ---
AUTOTUNE = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)

# --- Result storage ---
all_runs_results = []

print(f"Strategy   : {STRATEGY_LABEL}  (key={STRATEGY_KEY})")
print(f"Loss       : AdaptiveCBFocalLoss v2  (gamma={FOCAL_GAMMA}, beta={CB_BETA}, tau={ADAPTIVE_TAU})")
print(f"2-Phase    : Phase1={PHASE1_EPOCHS} epochs (static) + Phase2={EPOCHS-PHASE1_EPOCHS} epochs (adaptive)")
print(f"Stability  : min_recall_gap={MIN_RECALL_GAP}, factor_clamp=[0.5, 2.0]")
print(f"Unfreeze   : {UNFREEZE_BLOCKS}  |  Aug: {USE_AUG}  |  LR: {LR}")
print(f"Dropout    : {DROPOUT_RATE}  |  Label smoothing: {LABEL_SMOOTHING}")
print(f"Dataset    : {DATA_DIR}")
print(f"Output dir : {BASE_RESULT_DIR}")
print(f"Batch/Epochs: {BATCH_SIZE} / {EPOCHS}  |  Patience: {PATIENCE}")
print(f"N_RUNS     : {N_RUNS}  |  Seeds: {RANDOM_SEEDS[:N_RUNS]}")
print(f"FSDA       : reduction={FSDA_REDUCTION}  spatial_ks={FSDA_SPATIAL_KS}")

In [ ]:
# ========== 5. HELPER FUNCTIONS ========== #

efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def apply_freeze_strategy(base, unfreeze_blocks):
    """Selectively unfreeze EfficientNetB4 blocks (BN always frozen)."""
    base.trainable = False
    if unfreeze_blocks == "all":
        for layer in base.layers:
            if not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True
    elif isinstance(unfreeze_blocks, list) and len(unfreeze_blocks) > 0:
        for layer in base.layers:
            for block_num in unfreeze_blocks:
                if layer.name.startswith(f"block{block_num}"):
                    if not isinstance(layer, tf.keras.layers.BatchNormalization):
                        layer.trainable = True
                    break
    trainable = sum(1 for l in base.layers if l.trainable)
    total     = len(base.layers)
    print(f"  [{STRATEGY_LABEL}] {trainable}/{total} base layers trainable (BN frozen)")


def _collect_samples(split_dir, class_to_idx):
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None, use_aug=True):
    class_names  = sorted([
        d for d in os.listdir(os.path.join(data_dir, 'train'))
        if os.path.isdir(os.path.join(data_dir, 'train', d))
    ])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes  = len(class_names)
    h, w         = input_shape[:2]

    def load_and_preprocess(path, label):
        raw   = tf.io.read_file(path)
        img   = tf.image.decode_jpeg(raw, channels=3)
        img   = tf.image.resize(img, [h, w])
        img   = tf.cast(img, tf.float32)
        img   = efficientnet_preprocess(img)
        label = tf.one_hot(label, depth=num_classes)
        return img, label

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False, apply_augmentation=False):
        sdir               = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        n  = len(paths)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(n, seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if apply_augmentation:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training)
        ds = ds.prefetch(AUTOTUNE)
        return ds, n, fns, labels

    train_ds, n_train, _,           train_lbl  = _make_split('train', training=True,  apply_augmentation=use_aug)
    val_ds,   n_val,   _,           _          = _make_split('val',   training=False, apply_augmentation=False)
    test_ds,  n_test,  test_fnames, test_lbl   = _make_split('test',  training=False, apply_augmentation=False)

    cw      = class_weight.compute_class_weight(
        'balanced', classes=np.unique(train_lbl), y=train_lbl)
    cw_dict = dict(enumerate(cw))

    meta = SimpleNamespace(
        class_names       = class_names,
        class_to_idx      = class_to_idx,
        num_classes       = num_classes,
        test_filenames    = test_fnames,
        test_classes      = np.array(test_lbl),
        n_train           = n_train,
        n_val             = n_val,
        n_test            = n_test,
        class_weight_dict = cw_dict,
    )
    print(f"  Aug: {'ON' if use_aug else 'OFF'}  |  train={n_train}  val={n_val}  test={n_test}")
    return train_ds, val_ds, test_ds, meta


print("Helper functions defined.")

In [ ]:
# ========== 6. FSDA ARCHITECTURE + MODEL BUILDER ========== #

class FrequencyChannelAttention(tf.keras.layers.Layer):
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        r = max(C // self.reduction, 8)
        self.fc1 = Dense(r, use_bias=False, dtype='float32', name=f'{self.name}_fc1')
        self.fc2 = Dense(C, use_bias=False, dtype='float32', name=f'{self.name}_fc2')
        self.fc1.build((None, C))
        self.fc2.build((None, r))
        super().build(input_shape)

    def call(self, x, training=False):
        x_f32     = tf.cast(x, tf.float32)
        x_t       = tf.transpose(x_f32, [0, 3, 1, 2])
        x_complex = tf.complex(x_t, tf.zeros_like(x_t))
        x_fft     = tf.signal.fft2d(x_complex)
        mag       = tf.math.log1p(tf.abs(x_fft))
        freq_desc = tf.reduce_mean(mag, axis=[2, 3])
        attn = tf.nn.relu(self.fc1(freq_desc))
        attn = tf.nn.sigmoid(self.fc2(attn))
        attn = tf.reshape(attn, [tf.shape(x_f32)[0], 1, 1, tf.shape(x_f32)[3]])
        out  = x_f32 * attn
        return tf.cast(out, x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg['reduction'] = self.reduction
        return cfg


class FSDABlock(tf.keras.layers.Layer):
    def __init__(self, reduction=16, spatial_kernel=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction      = reduction
        self.spatial_kernel = spatial_kernel

    def build(self, input_shape):
        self.freq_attn = FrequencyChannelAttention(
            self.reduction, name=f'{self.name}_freq_attn')
        self.sp_conv = Conv2D(
            1, self.spatial_kernel, padding='same', use_bias=False,
            kernel_initializer='glorot_uniform', dtype='float32',
            name=f'{self.name}_sp_conv')
        self.bn = BatchNormalization(dtype='float32', name=f'{self.name}_bn')
        self.freq_attn.build(input_shape)
        sp_input_shape = tuple(input_shape[:-1]) + (2,)
        self.sp_conv.build(sp_input_shape)
        self.bn.build(input_shape)
        super().build(input_shape)

    def call(self, x, training=False):
        input_dtype = x.dtype
        freq_out = tf.cast(self.freq_attn(x, training=training), tf.float32)
        x_f32    = tf.cast(x, tf.float32)
        avg_pool = tf.reduce_mean(x_f32, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x_f32,  axis=-1, keepdims=True)
        sp_attn  = tf.nn.sigmoid(
            self.sp_conv(tf.concat([avg_pool, max_pool], axis=-1))
        )
        spatial_out = x_f32 * sp_attn
        fused = freq_out + spatial_out
        fused = self.bn(fused, training=training)
        fused = tf.cast(fused, input_dtype)
        return fused, sp_attn

    def compute_output_spec(self, x, training=False):
        import keras
        sp_shape = tuple(x.shape[:-1]) + (1,)
        return (
            keras.KerasTensor(x.shape, dtype=x.dtype),
            keras.KerasTensor(sp_shape, dtype='float32'),
        )

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction': self.reduction, 'spatial_kernel': self.spatial_kernel})
        return cfg


CUSTOM_OBJECTS = {
    'FrequencyChannelAttention':         FrequencyChannelAttention,
    'FSDABlock':                         FSDABlock,
    'AdaptiveClassBalancedFocalLossV2':  AdaptiveClassBalancedFocalLossV2,
}


def build_fsda_model(input_shape, num_classes, steps_per_epoch, samples_per_class):
    """Build EfficientNetB4 + FSDA + Adaptive CB Focal Loss v2."""
    base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=input_shape)
    apply_freeze_strategy(base, UNFREEZE_BLOCKS)

    feat_map = base.output
    attended, sp_attn_map = FSDABlock(
        reduction=FSDA_REDUCTION, spatial_kernel=FSDA_SPATIAL_KS, name='fsda'
    )(feat_map)

    x   = GlobalAveragePooling2D(name='gap')(attended)
    x   = BatchNormalization(name='head_bn')(x)
    x   = Dense(256, activation='relu', kernel_regularizer=l2(1e-5), name='head_dense')(x)
    x   = Dropout(DROPOUT_RATE, name='head_dropout')(x)
    out = Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)

    model = Model(inputs=base.input, outputs=out, name='EfficientNetB4_FSDA_AdaptiveCB_v2')
    vis_model = Model(inputs=base.input, outputs=[out, sp_attn_map],
                      name='EfficientNetB4_FSDA_AdaptiveCB_v2_vis')

    loss_fn = AdaptiveClassBalancedFocalLossV2(
        samples_per_class=samples_per_class,
        num_classes=num_classes,
        gamma=FOCAL_GAMMA,
        beta=CB_BETA,
        label_smoothing=LABEL_SMOOTHING,
    )

    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR,
        decay_steps=steps_per_epoch * 5,
        decay_rate=0.9,
        staircase=True,
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss=loss_fn,
        metrics=['accuracy'],
    )
    return model, vis_model, loss_fn


print("FSDA + Adaptive CB v2 architecture defined.")

In [ ]:
# ========== 7. MULTI-RUN TRAINING (2-PHASE) ========== #

for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print("\n" + "="*70)
    print(f" RUN {run_idx+1}/{N_RUNS}  --  seed={seed}")
    print(f" Strategy : {STRATEGY_LABEL}")
    print(f" 2-Phase  : {PHASE1_EPOCHS} static + {EPOCHS-PHASE1_EPOCHS} adaptive")
    print("="*70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    # ── Build datasets ─────────────────────────────────────────────────────
    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed, use_aug=USE_AUG)
    steps_per_epoch = meta.n_train // BATCH_SIZE

    # ── Compute samples per class ──────────────────────────────────────────
    samples_per_class = np.array([
        len([f for f in os.listdir(os.path.join(DATA_DIR, 'train', cn))
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))])
        for cn in meta.class_names
    ])
    print(f"  Samples/class: {dict(zip(meta.class_names, samples_per_class))}")

    # ── Build model ────────────────────────────────────────────────────────
    model, vis_model, loss_fn = build_fsda_model(
        INPUT_SHAPE, meta.num_classes, steps_per_epoch, samples_per_class)
    print(f"  Trainable params: {model.count_params():,}")

    # ── 2-Phase Adaptive Callback ──────────────────────────────────────────
    adaptive_cb = TwoPhaseAdaptiveCallback(
        loss_fn=loss_fn,
        val_ds=val_ds,
        num_classes=meta.num_classes,
        class_names=meta.class_names,
        phase1_epochs=PHASE1_EPOCHS,
        tau=ADAPTIVE_TAU,
        min_recall_gap=MIN_RECALL_GAP,
    )

    callbacks = [
        adaptive_cb,
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv'), append=False),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'best_model.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ]

    # NOTE: class_weight RE-ENABLED (was removed in v1)
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=meta.class_weight_dict,
        callbacks=callbacks,
    )

    # ── Save adaptive weight history ───────────────────────────────────────
    weight_history_df = pd.DataFrame([
        {'epoch': h['epoch'], 'phase': h['phase'],
         **{f'recall_{cn}': h['per_class_recall'][i] for i, cn in enumerate(meta.class_names)},
         **{f'factor_{cn}': h['adaptive_factor'][i] for i, cn in enumerate(meta.class_names)}}
        for h in adaptive_cb.history
    ])
    weight_history_df.to_csv(os.path.join(RESULT_DIR, 'adaptive_weight_history.csv'), index=False)

    # ── Learning curves (3 panels) ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Val')
    axes[0].axvline(PHASE1_EPOCHS, color='red', linestyle='--', alpha=0.7, label='Phase 2 start')
    axes[0].set_title(f'Accuracy -- Run {run_idx+1}'); axes[0].legend(); axes[0].grid(alpha=0.3)
    
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Val')
    axes[1].axvline(PHASE1_EPOCHS, color='red', linestyle='--', alpha=0.7, label='Phase 2 start')
    axes[1].set_title(f'Loss -- Run {run_idx+1}'); axes[1].legend(); axes[1].grid(alpha=0.3)
    
    for cn in meta.class_names:
        axes[2].plot(weight_history_df['epoch'], weight_history_df[f'factor_{cn}'], label=cn)
    axes[2].axvline(PHASE1_EPOCHS, color='red', linestyle='--', alpha=0.7)
    axes[2].axhline(1.0, color='black', linestyle=':', alpha=0.5)
    axes[2].set_title('Adaptive Factors'); axes[2].legend(); axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curve.png'), dpi=300)
    plt.show()

    # ── Evaluate ───────────────────────────────────────────────────────────
    best_model = load_model(
        os.path.join(RESULT_DIR, 'best_model.keras'),
        custom_objects=CUSTOM_OBJECTS,
    )
    pred_probs = best_model.predict(test_ds, verbose=0)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes
    class_names = meta.class_names

    report = classification_report(
        y_true_run, y_pred_run, target_names=class_names, output_dict=True, digits=4)
    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(
            y_true_run, y_pred_run, target_names=class_names, digits=4))

    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_names, yticklabels=class_names, cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix -- Run {run_idx+1} (seed={seed})')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.close()

    test_acc = np.mean(y_pred_run == y_true_run)

    all_runs_results.append({
        'run':               run_idx + 1,
        'seed':              seed,
        'accuracy':          test_acc,
        'precision':         report['weighted avg']['precision'],
        'recall':            report['weighted avg']['recall'],
        'f1_score':          report['weighted avg']['f1-score'],
        'per_class_metrics': {c: {'precision': report[c]['precision'],
                                   'recall':    report[c]['recall'],
                                   'f1-score':  report[c]['f1-score']}
                               for c in class_names},
        'result_dir':        RESULT_DIR,
        'history':           history.history,
        'y_true':            y_true_run,
        'y_pred':            y_pred_run,
        'pred_probs':        pred_probs,
        'class_names':       class_names,
        'test_filenames':    meta.test_filenames,
        'n_train':           meta.n_train,
        'n_val':             meta.n_val,
        'n_test':            meta.n_test,
        'adaptive_history':  adaptive_cb.history,
    })

    print(f"\n  Acc={test_acc:.4f}  "
          f"P={report['weighted avg']['precision']:.4f}  "
          f"R={report['weighted avg']['recall']:.4f}  "
          f"F1={report['weighted avg']['f1-score']:.4f}")

    tf.keras.backend.clear_session()

# ── Save summary CSV ───────────────────────────────────────────────────────
summary_path = os.path.join(BASE_RESULT_DIR, 'strategy_summary.csv')
with open(summary_path, 'w', newline='') as csvf:
    fieldnames = ['strategy_key', 'strategy_label', 'unfreeze_blocks', 'lr',
                  'use_aug', 'fsda_reduction', 'fsda_spatial_ks',
                  'adaptive_tau', 'focal_gamma', 'cb_beta', 'phase1_epochs',
                  'run', 'seed', 'accuracy', 'precision', 'recall', 'f1_score']
    writer = csv.DictWriter(csvf, fieldnames=fieldnames)
    writer.writeheader()
    for r in all_runs_results:
        writer.writerow({
            'strategy_key':    STRATEGY_KEY,
            'strategy_label':  STRATEGY_LABEL,
            'unfreeze_blocks': str(UNFREEZE_BLOCKS),
            'lr':              LR,
            'use_aug':         USE_AUG,
            'fsda_reduction':  FSDA_REDUCTION,
            'fsda_spatial_ks': FSDA_SPATIAL_KS,
            'adaptive_tau':    ADAPTIVE_TAU,
            'focal_gamma':     FOCAL_GAMMA,
            'cb_beta':         CB_BETA,
            'phase1_epochs':   PHASE1_EPOCHS,
            'run':             r['run'],
            'seed':            r['seed'],
            'accuracy':        r['accuracy'],
            'precision':       r['precision'],
            'recall':          r['recall'],
            'f1_score':        r['f1_score'],
        })

print("\n" + "="*70)
print(f" ALL {N_RUNS} RUNS COMPLETED -- {STRATEGY_LABEL}")
print(f" Saved -> {summary_path}")
print("="*70)

---
## Section 2 — Results Aggregation & Analysis

In [ ]:
# ========== AGGREGATE RESULTS ========== #
print("\n" + "="*80)
print("AGGREGATING RESULTS FROM ALL RUNS")
print("="*80 + "\n")

accuracies = [r['accuracy'] for r in all_runs_results]
precisions = [r['precision'] for r in all_runs_results]
recalls    = [r['recall'] for r in all_runs_results]
f1_scores  = [r['f1_score'] for r in all_runs_results]

overall_stats = {
    'Accuracy':  {'mean': np.mean(accuracies),  'std': np.std(accuracies),  'values': accuracies},
    'Precision': {'mean': np.mean(precisions),  'std': np.std(precisions),  'values': precisions},
    'Recall':    {'mean': np.mean(recalls),     'std': np.std(recalls),     'values': recalls},
    'F1-Score':  {'mean': np.mean(f1_scores),   'std': np.std(f1_scores),   'values': f1_scores},
}

print("OVERALL METRICS:")
print("-" * 60)
for metric_name, stats in overall_stats.items():
    print(f"{metric_name:12s}: {stats['mean']:.4f} +/- {stats['std']:.4f}")
    print(f"              Runs: {[f'{v:.4f}' for v in stats['values']]}")
print("-" * 60)

# Per-class stats
class_names = all_runs_results[0]['class_names']
per_class_stats = {}
for cn in class_names:
    per_class_stats[cn] = {}
    for metric in ['precision', 'recall', 'f1-score']:
        values = [r['per_class_metrics'][cn][metric] for r in all_runs_results]
        per_class_stats[cn][metric] = {'mean': np.mean(values), 'std': np.std(values), 'values': values}

print("\nPER-CLASS F1-SCORE:")
for cn in class_names:
    s = per_class_stats[cn]['f1-score']
    print(f"  {cn:<30} {s['mean']:.4f} +/- {s['std']:.4f}")

# Additional metrics
print("\nADDITIONAL METRICS:")
for r in all_runs_results:
    kappa   = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc     = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    print(f"  Run {r['run']}: BalAcc={bal_acc:.4f}  Kappa={kappa:.4f}  MCC={mcc:.4f}")

# Comparison with v1
print("\n" + "="*60)
print("COMPARISON WITH v1:")
print(f"  v1 F1: 0.9064 +/- 0.0137")
print(f"  v2 F1: {np.mean(f1_scores):.4f} +/- {np.std(f1_scores):.4f}")
improvement = (np.mean(f1_scores) - 0.9064) * 100
print(f"  Improvement: {improvement:+.2f} percentage points")
print("="*60)

In [ ]:
# ========== ADAPTIVE WEIGHT EVOLUTION VISUALIZATION ========== #

n_runs_plot = len(all_runs_results)
fig, axes = plt.subplots(n_runs_plot, 2, figsize=(14, 5 * n_runs_plot))
if n_runs_plot == 1:
    axes = axes[np.newaxis, :]

for run_idx, r in enumerate(all_runs_results):
    adaptive_hist = r['adaptive_history']
    epochs_list = [h['epoch'] for h in adaptive_hist]
    cn_list = r['class_names']
    
    # Left: Per-class recall
    for ci, cn in enumerate(cn_list):
        recall_vals = [h['per_class_recall'][ci] for h in adaptive_hist]
        axes[run_idx, 0].plot(epochs_list, recall_vals, marker='o', markersize=3, label=cn)
    axes[run_idx, 0].axvline(PHASE1_EPOCHS, color='red', linestyle='--', alpha=0.7, label='Phase 2')
    axes[run_idx, 0].set_title(f'Val Recall (Run {r["run"]})', fontweight='bold')
    axes[run_idx, 0].set_xlabel('Epoch'); axes[run_idx, 0].set_ylabel('Recall')
    axes[run_idx, 0].legend(); axes[run_idx, 0].grid(alpha=0.3)
    axes[run_idx, 0].set_ylim([0, 1.05])
    
    # Right: Adaptive factors
    for ci, cn in enumerate(cn_list):
        factor_vals = [h['adaptive_factor'][ci] for h in adaptive_hist]
        axes[run_idx, 1].plot(epochs_list, factor_vals, marker='s', markersize=3, label=cn)
    axes[run_idx, 1].axvline(PHASE1_EPOCHS, color='red', linestyle='--', alpha=0.7, label='Phase 2')
    axes[run_idx, 1].axhline(1.0, color='black', linestyle=':', alpha=0.5)
    axes[run_idx, 1].set_title(f'Adaptive Factors (Run {r["run"]})', fontweight='bold')
    axes[run_idx, 1].set_xlabel('Epoch'); axes[run_idx, 1].set_ylabel('Factor')
    axes[run_idx, 1].legend(); axes[run_idx, 1].grid(alpha=0.3)

plt.suptitle('2-Phase Adaptive CB Loss v2 — Weight Evolution\n'
             'Phase 1: Static (stable convergence) | Phase 2: Adaptive (focus tuning)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'adaptive_weight_evolution_v2.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved -> adaptive_weight_evolution_v2.png")

In [ ]:
# ========== SAVE REPORT & ZIP ========== #

report_lines = []
report_lines.append("=" * 80)
report_lines.append("EfficientNetB4 + FSDA + ADAPTIVE CB LOSS v2 — EXPERIMENT REPORT")
report_lines.append("=" * 80)
report_lines.append(f"Strategy: {STRATEGY_LABEL}")
report_lines.append(f"Key: {STRATEGY_KEY}")
report_lines.append(f"")
report_lines.append(f"IMPROVEMENTS OVER v1:")
report_lines.append(f"  - 2-Phase training: {PHASE1_EPOCHS} static + {EPOCHS-PHASE1_EPOCHS} adaptive")
report_lines.append(f"  - Conservative tau: {ADAPTIVE_TAU} (was 0.3)")
report_lines.append(f"  - Softer focal gamma: {FOCAL_GAMMA} (was 2.0)")
report_lines.append(f"  - Lower beta: {CB_BETA} (was 0.9999)")
report_lines.append(f"  - Label smoothing: {LABEL_SMOOTHING}")
report_lines.append(f"  - Stability guard: min_recall_gap={MIN_RECALL_GAP}")
report_lines.append(f"  - Factor clamping: [0.5, 2.0]")
report_lines.append(f"  - class_weight re-enabled in model.fit()")
report_lines.append(f"  - Dropout: {DROPOUT_RATE} (was 0.5)")
report_lines.append(f"")
report_lines.append(f"OVERALL PERFORMANCE (Mean +/- Std):")
report_lines.append("-" * 60)
for metric, stats in overall_stats.items():
    report_lines.append(f"  {metric:<12}: {stats['mean']:.4f} +/- {stats['std']:.4f}")
report_lines.append(f"")
report_lines.append(f"PER-CLASS F1-SCORE:")
for cn in class_names:
    s = per_class_stats[cn]['f1-score']
    report_lines.append(f"  {cn:<30} {s['mean']:.4f} +/- {s['std']:.4f}")
report_lines.append(f"")
report_lines.append(f"COMPARISON WITH v1:")
report_lines.append(f"  v1 F1: 0.9064 +/- 0.0137")
report_lines.append(f"  v2 F1: {np.mean(f1_scores):.4f} +/- {np.std(f1_scores):.4f}")

report_text = "\n".join(report_lines)
print(report_text)

with open(os.path.join(BASE_RESULT_DIR, 'EXPERIMENT_REPORT_v2.txt'), 'w', encoding='utf-8') as f:
    f.write(report_text)

# ZIP
zip_path = "/kaggle/working/EfficientNetB4_FSDA_AdaptiveCB_v2_Complete"
shutil.make_archive(zip_path, 'zip', BASE_RESULT_DIR)
zip_size = os.path.getsize(f"{zip_path}.zip") / (1024*1024)
print(f"\nArchive: {zip_path}.zip  ({zip_size:.2f} MB)")
print("DONE!")